# Finetune YOLO11s-seg for instance segmentation

Walkthrough of the two scripts, which can also be run from the terminal:

1. `dataset/build_dataset.py`: check the labelled images in `dataset/source/` and split them into train/val/test
2. `perception/finetune.py`: train from the pretrained weights and score on the test split

The images and polygon labels are made elsewhere (rendered views of the 3D models, or a CVAT
"YOLO segmentation" export). Put them in `dataset/source/` with `images/`, `labels/` and `classes.txt`.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "perception" else Path.cwd()  # repository root
sys.path[:0] = [str(ROOT / "dataset"), str(ROOT / "perception")]
import build_dataset
import finetune

## 1. Build the dataset

In [ ]:
build_dataset.build(
    source=build_dataset.SOURCE_DIR,
    ratios=(0.7, 0.2, 0.1),  # train, val, test
    seed=0,
    by_folder=False,         # True keeps each folder under images/ in one split
    preview=True,
    overwrite=False,         # set True to delete the earlier build and redo it
)

## 2. Check the outlines

In [ ]:
all_previews = sorted((build_dataset.DATASET_DIR / "preview").glob("*.jpg"))
previews = all_previews[::max(1, len(all_previews) // 12)][:12]  # 12 spread across the set
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax in axes.flat:
    ax.axis("off")
for ax, path in zip(axes.flat, previews):
    ax.imshow(cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB))
    ax.set_title(path.name, fontsize=8)
plt.tight_layout()

## 3. Finetune

In [ ]:
out = finetune.finetune(
    epochs=50,
    imgsz=640,
    batch=8,       # lower if the GPU runs out of memory
    freeze=15,     # first 15 layers kept fixed
    patience=15,   # stop early after 15 epochs without improvement
)

## 4. Try the finetuned model on a test image

In [ ]:
from ultralytics import YOLO

model = YOLO(str(out))
test_images = sorted((build_dataset.DATASET_DIR / "test" / "images").iterdir())
result = model.predict(str(test_images[0]), verbose=False)[0]
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")

Run the perception program with the new weights from the repository root:

```bash
python perception/main.py --model models/yolo11s-seg-custom.pt
```